In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# 1. Explorar los datos
def explorar_datos_salud():
    """Carga y analiza los datos de salud cardíaca"""
    X, y = make_classification(n_samples=1000, n_features=8,
                               n_informative=5, n_classes=2,
                               random_state=42)

    feature_names = ['Edad', 'Presión Arterial', 'Colesterol', 'Glucosa',
                     'IMC', 'Fumador', 'Actividad Física', 'Historial Familiar']

    print("=== Análisis de Datos de Salud Cardíaca ===")
    print(f"Muestras totales: {X.shape[0]}")
    print(f"Características clínicas: {X.shape[1]}")
    print(f"Pacientes sanos (0): {sum(y==0)}")
    print(f"Pacientes con riesgo (1): {sum(y==1)}\n")
    
    return X, y, feature_names

# 2. Construcción y Evaluación de Modelos
def construir_y_evaluar_modelos(X, y):
    """Divide los datos y entrena modelos de clasificación"""
    # División de datos: 80% entrenamiento, 20% prueba
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Escalado de datos 
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    modelos = {
        "Regresión Logística": LogisticRegression(random_state=42),
        "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
    }
    
    resultados = {}
    
    for nombre, modelo in modelos.items():
        print(f"--- Entrenando: {nombre} ---")
        if nombre == "Regresión Logística":
            modelo.fit(X_train_scaled, y_train)
            y_pred = modelo.predict(X_test_scaled)
        else:
            modelo.fit(X_train, y_train)
            y_pred = modelo.predict(X_test)
            
        acc = accuracy_score(y_test, y_pred)
        resultados[nombre] = {
            "modelo": modelo,
            "accuracy": acc,
            "y_pred": y_pred,
            "y_test": y_test
        }
        
        print(f"Precisión (Accuracy): {acc:.4f}")
        print("Reporte de Clasificación:")
        print(classification_report(y_test, y_pred))
        print("-" * 40)
        
    return resultados, X_test, y_test

# 3. Interpretar resultados y Factores clave
def interpretar_resultados(resultados, feature_names):
    """Analiza la importancia de las características según los modelos"""
    print("=== Interpretación de Resultados ===")
    
    # Importancia con Random Forest
    rf_model = resultados["Random Forest"]["modelo"]
    importancias = rf_model.feature_importances_
    
    df_importancia = pd.DataFrame({
        'Característica': feature_names,
        'Importancia': importancias
    }).sort_values(by='Importancia', ascending=False)
    
    print("\nFactores más importantes (según Random Forest):")
    print(df_importancia.to_string(index=False))
    
    # Coeficientes con Regresión Logística
    lr_model = resultados["Regresión Logística"]["modelo"]
    coeficientes = lr_model.coef_[0]
    
    df_coef = pd.DataFrame({
        'Característica': feature_names,
        'Coeficiente': coeficientes
    }).sort_values(by='Coeficiente', key=abs, ascending=False)
    
    print("\nImpacto de las características (según Regresión Logística):")
    print(df_coef.to_string(index=False))

# --- Ejecución principal del script ---
if __name__ == "__main__":
    X, y, feature_names = explorar_datos_salud()
    resultados, X_test, y_test = construir_y_evaluar_modelos(X, y)
    interpretar_resultados(resultados, feature_names)

=== Análisis de Datos de Salud Cardíaca ===
Muestras totales: 1000
Características clínicas: 8
Pacientes sanos (0): 498
Pacientes con riesgo (1): 502

--- Entrenando: Regresión Logística ---
Precisión (Accuracy): 0.8150
Reporte de Clasificación:
              precision    recall  f1-score   support

           0       0.80      0.81      0.81        95
           1       0.83      0.82      0.82       105

    accuracy                           0.81       200
   macro avg       0.81      0.81      0.81       200
weighted avg       0.82      0.81      0.82       200

----------------------------------------
--- Entrenando: Random Forest ---
Precisión (Accuracy): 0.9350
Reporte de Clasificación:
              precision    recall  f1-score   support

           0       0.94      0.93      0.93        95
           1       0.93      0.94      0.94       105

    accuracy                           0.94       200
   macro avg       0.94      0.93      0.93       200
weighted avg       0.94  